<h4 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
ساخت مدل طبقه‌بندی
</font>
</h4>
<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در این بخش یک مدل یادگیری عمیق برای طبقه‌بندی تصاویر گربه و سگ می‌سازیم. می‌توانید از یک مدل CNN ساده یا مدل انتقال یادگیری (مانند MobileNetV2) استفاده کنید. مدل باید یک خروجی با مقدار احتمال (بین ۰ و ۱) داشته باشد.
</font>
</p>

<details>
<summary>راهنمایی بیشتر</summary>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<li>در مدل CNN ساده، از چند لایه Conv2D و MaxPooling2D استفاده کنید و در انتها یک لایه Dense با activation='sigmoid' قرار دهید.</li>
<li>در مدل انتقال یادگیری، ابتدا base_model را فریز کنید و فقط لایه‌های بالایی را آموزش دهید.</li>
<li>برای کاهش احتمال بیش‌برازش، می‌توانید Dropout اضافه کنید.</li>
<li>خروجی مدل باید یک مقدار بین ۰ و ۱ باشد (برای binary classification).</li>
</ul>
</details>



<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
گربه یا سگ؟ مسئله این است!
</font>
</h1>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
در این پروژه، هدف ساخت یک مدل یادگیری عمیق برای تشخیص تصاویر گربه و سگ است. مراحل کار به صورت گام‌به‌گام و با توضیحات دقیق ارائه شده تا هم با مفاهیم و هم با پیاده‌سازی عملی آشنا شوید.
</font>
</p>

<h4 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
۱. ساختار داده و مسیرها
</font>
</h4>
<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
داده‌ها در دو پوشه <code>train/</code> و <code>test/</code> قرار دارند. در <b>train/</b> دو زیرپوشه <b>cats/</b> و <b>dogs/</b> وجود دارد که هرکدام شامل تصاویر مربوط به همان کلاس هستند. پوشه <b>test/</b> شامل تصاویر بدون برچسب است. نام پوشه، برچسب هر تصویر را مشخص می‌کند.
<br>نمونه ساختار:
<pre>project/
  ├── train/
  │     ├── cats/
  │     └── dogs/
  └── test/
</pre>
</font>
</p>


In [ ]:
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
import pandas as pd
import matplotlib.pyplot as plt


<h4 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
۲. بارگذاری و آماده‌سازی داده‌ها
</font>
</h4>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <li>تصاویر را از پوشه‌های <b>cats</b> و <b>dogs</b> می‌خوانیم.</li>
    <li>اندازه تصاویر را به مقدار ثابت (مثلاً <b>۱۲۸×۱۲۸</b>) تغییر می‌دهیم.</li>
    <li>مقادیر پیکسل را بین <b>۰ و ۱</b> نرمال‌سازی می‌کنیم.</li>
    <li>برچسب هر تصویر بر اساس نام پوشه تعیین می‌شود (<b>cat=۰</b>، <b>dog=۱</b>).</li>
    <li>داده‌ها را با <code>train_test_split</code> به آموزش و تست تقسیم می‌کنیم.</li>
    <li><b>نکته:</b> اگر داده‌ها نامتوازن هستند، از پارامتر <code>stratify</code> استفاده کنید تا نسبت کلاس‌ها در آموزش و تست حفظ شود.</li>
</ul>

<details>
<summary>راهنمایی بیشتر</summary>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <li>برای خواندن تصویر از دستور <code>cv2.imread(path)</code> استفاده کنید که تصویر را به صورت آرایه عددی می‌خواند.</li>
    <li>برای تغییر اندازه تصویر از <code>cv2.resize(image, (128, 128))</code> استفاده کنید.</li>
    <li>برای نرمال‌سازی کافی است آرایه تصویر را بر ۲۵۵ تقسیم کنید: <code>img = img / 255.0</code></li>
    <li>برچسب هر تصویر را با توجه به نام پوشه تعیین کنید: اگر تصویر در پوشه <b>cats</b> بود مقدار ۰ و اگر در <b>dogs</b> بود مقدار ۱ بدهید.</li>
    <li>در نهایت داده‌ها را با <code>train_test_split(X, y, test_size=..., stratify=y)</code> به آموزش و تست تقسیم کنید.</li>
</ul>
</details>


In [ ]:
#To_do

In [ ]:
!unzip Cat\ OR\ Dog2\ \(2\).zip

In [ ]:
IMG_SIZE = 128
data = []
labels = []
train_dir = 'Data/train' # Corrected the directory path

for category in ['cats', 'dogs']:
    path = os.path.join(train_dir, category)
    class_num = 0 if category == 'cats' else 1
    for img_name in os.listdir(path):
        try:
            img_array = cv2.imread(os.path.join(path, img_name))
            new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
            data.append(new_array / 255.0)
            labels.append(class_num)
        except Exception as e:
            pass # Handle corrupted images

X = np.array(data)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Data loading and preprocessing complete.")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

<h4 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
ساخت مدل طبقه‌بندی
</font>
</h4>
<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در این بخش یک مدل یادگیری عمیق برای طبقه‌بندی تصاویر گربه و سگ می‌سازیم. می‌توانید از یک مدل CNN ساده یا مدل انتقال یادگیری (مانند MobileNetV2) استفاده کنید. مدل باید یک خروجی با مقدار احتمال (بین ۰ و ۱) داشته باشد.
</font>
</p>

<details>
<summary>راهنمایی بیشتر</summary>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<li>در مدل CNN ساده، از چند لایه Conv2D و MaxPooling2D استفاده کنید و در انتها یک لایه Dense با activation='sigmoid' قرار دهید.</li>
<li>در مدل انتقال یادگیری، ابتدا base_model را فریز کنید و فقط لایه‌های بالایی را آموزش دهید.</li>
<li>برای کاهش احتمال بیش‌برازش، می‌توانید Dropout اضافه کنید.</li>
<li>خروجی مدل باید یک مقدار بین ۰ و ۱ باشد (برای binary classification).</li>
</ul>
</details>



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2

# Load the MobileNetV2 model pre-trained on ImageNet
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                         include_top=False, # Do not include the ImageNet classifier at the top
                         weights='imagenet')

# Freeze the base model layers
base_model.trainable = False

# Create a new model on top of the base model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(), # Add a global average pooling layer
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

<h4 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
آموزش مدل و تنظیمات Training
</font>
</h4>
<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در این مرحله مدل را با داده‌های آموزش آموزش می‌دهیم. می‌توانید از EarlyStopping برای جلوگیری از بیش‌برازش استفاده کنید. همچنین می‌توانید از ImageDataGenerator برای افزایش داده‌ها (augmentation) بهره ببرید.
</font>
</p>

<details>
<summary>راهنمایی بیشتر</summary>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<li>برای آموزش مدل از <code>fit</code> یا <code>fit_generator</code> استفاده کنید.</li>
<li>برای جلوگیری از overfitting، EarlyStopping را با پارامتر patience مناسب فعال کنید.</li>
<li>برای افزایش داده‌ها، از ImageDataGenerator با پارامترهای چرخش، جابجایی و ... استفاده کنید.</li>
<li>برای ارزیابی مدل، داده‌های validation را مشخص کنید.</li>
</ul>
</details>

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Define EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=50, # You can adjust the number of epochs
    batch_size=32,
    validation_split=0.2, # Using a validation split from the training data
    callbacks=[early_stopping]
)

<h4 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
ارزیابی مدل و نمایش نتایج
</font>
</h4>
<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در این بخش عملکرد مدل را روی داده‌های تست بررسی می‌کنیم. معیار ارزیابی مدل <b>accuracy score</b> است و مدل باید دقت حداقل <b>۵۰٪</b> را به دست بیاورد تا امتیاز کامل را کسب کند. همچنین می‌توانید چند نمونه تصویر تست و پیش‌بینی مدل را کنار هم نمایش دهید تا کیفیت مدل را به صورت بصری ارزیابی کنید.
</font>
</p>

<details>
<summary>راهنمایی بیشتر</summary>
<ul dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<li>برای محاسبه دقت از <code>sklearn.metrics.accuracy_score</code> استفاده کنید.</li>
<li>برای تبدیل خروجی مدل به برچسب نهایی، مقدار احتمال را با آستانه ۰.۵ به ۰ یا ۱ تبدیل کنید.</li>
<li>اگر مدل شما خروجی sigmoid دارد، خروجی‌های بزرگ‌تر از ۰.۵ را به کلاس ۱ و بقیه را به کلاس ۰ نسبت دهید.</li>
</ul>
</details>

In [ ]:
from sklearn.metrics import accuracy_score

# Predict probabilities on the test set
y_pred_prob = model.predict(X_test)

# Convert probabilities to binary class labels (0 or 1)
y_pred = (y_pred_prob > 0.5).astype(int)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy on the test data: {accuracy:.4f}")

<h4 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
ساخت فایل خروجی (Submission)
</font>
</h4>
<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در این بخش تصاویر تست را بارگذاری می‌کنیم و با استفاده از مدل آموزش‌دیده، برچسب هر تصویر را پیش‌بینی می‌کنیم. سپس یک DataFrame به نام <code>submission</code> می‌سازیم که شامل دو ستون است:
<ul>
<li><b>image_number:</b> شماره یا نام تصویر (مثلاً اگر نام فایل <code>1234.jpg</code> باشد، مقدار این ستون <code>1234</code> خواهد بود.)</li>
<li><b>category:</b> برچسب پیش‌بینی‌شده برای هر تصویر (<code>cat</code> یا <code>dog</code>)</li>
</ul>
در پایان، این DataFrame را به صورت فایل CSV ذخیره می‌کنیم تا برای ارسال یا ارزیابی مورد استفاده قرار گیرد.
</font>
</p>

In [ ]:
test_dir = 'Data/test'
test_images = []
image_numbers = []

for img_name in os.listdir(test_dir):
    try:
        img_path = os.path.join(test_dir, img_name)
        img_array = cv2.imread(img_path)
        new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
        test_images.append(new_array / 255.0)
        image_numbers.append(int(os.path.splitext(img_name)[0])) # Extract image number from filename
    except Exception as e:
        print(f"Error loading image {img_name}: {e}")

X_test_submission = np.array(test_images)

# Predict probabilities on the test submission set
submission_predictions_prob = model.predict(X_test_submission)

# Convert probabilities to binary class labels (0 or 1)
submission_predictions = (submission_predictions_prob > 0.5).astype(int)

# Convert class labels to 'cat' or 'dog'
category_labels = ['cat', 'dog']
predicted_categories = [category_labels[pred] for pred in submission_predictions.flatten()]

# Create submission DataFrame
submission = pd.DataFrame({
    'image_number': image_numbers,
    'Category': predicted_categories
})

# Sort by image_number to ensure correct order in submission file
submission = submission.sort_values(by='image_number').reset_index(drop=True)

print("Submission file created successfully.")
display(submission.head())

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول‌های زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد. همچنین اگر از گوگل کولب استفاده می‌کنید، در صورت نیاز به پشتیبانی حتماً آخرین نسخه از نت‌بوک را به‌صورت دستی دانلود کرده و داخل فایل ارسالی قرار دهید یا لینک کولب را با ما به‌اشتراک بگذارید.
</font>
</p>

In [ ]:
import zipfile
import joblib

if not os.path.exists(os.path.join(os.getcwd(), 'Cat_or_Dog_CNN.ipynb')):
    %notebook -e Cat_or_Dog_CNN.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['Cat_or_Dog_CNN.ipynb', 'submission.csv']
compress(file_names)